# 05 — Three-Era Convergence Analysis

Compares how quickly the field compresses in Year 1 of three regulation eras:
- **2014** — Hybrid Power Unit Era (Year 1): via Jolpica API
- **2022** — Ground Effect Era (Year 1): via FastF1
- **2026** — 2026 Era (Year 1, live): via FastF1

Core question: do convergence patterns repeat across regulation resets?
Metric: constructor points gap (P1-P10) and Gini coefficient, by round.

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd

import scripts.export_data as exp
from src.data.jolpica_client import get_constructor_standings_by_round
from src.analysis.championship_trajectory import cumulative_constructor_points
from src.analysis.era_convergence import convergence_by_round, era_comparison_table

## 1. Load 2014 standings (Jolpica API)

Fetches cumulative constructor points after each of the 19 rounds.
Cached in data/cache/jolpica/ — first run ~10s, subsequent runs instant.

In [2]:
standings_2014 = get_constructor_standings_by_round(2014)
print(f'2014: {len(standings_2014)} rows, {standings_2014["round"].nunique()} rounds, '
      f'{standings_2014["constructor_id"].nunique()} constructors')
standings_2014[["round","constructor_id","points","position"]].head(11)

2014: 209 rows, 19 rounds, 11 constructors


,round,constructor_id,points,position
0,1,mclaren,33.0,1.0
1,1,mercedes,25.0,2.0
2,1,ferrari,18.0,3.0
3,1,williams,10.0,4.0
4,1,force_india,9.0,5.0
5,1,toro_rosso,6.0,6.0
6,1,sauber,0.0,7.0
7,1,marussia,0.0,8.0
8,1,lotus_f1,0.0,NaN
9,1,caterham,0.0,NaN


## 2. Load 2022 standings (championship_trajectory parquet)

The championship_trajectory export holds 2022-2025 cumulative standings by round.

In [3]:
traj = exp.read('championship_trajectory')
standings_2022 = (
    traj[traj['season'] == 2022]
    .rename(columns={'cumulative_points': 'points'})
    [['season', 'round', 'constructor_id', 'constructor_name', 'points']]
    .copy()
)
print(f'2022: {len(standings_2022)} rows, {standings_2022["round"].nunique()} rounds, '
      f'{standings_2022["constructor_id"].nunique()} constructors')

2022: 220 rows, 22 rounds, 10 constructors


## 3. Build 2026 standings from results_2026 parquet

2026 data is stored as individual race results; cumulative standings are computed here.

In [4]:
results_2026 = exp.read('results_2026')
cumulative_2026 = cumulative_constructor_points(results_2026)
standings_2026 = (
    cumulative_2026
    .rename(columns={'cumulative_points': 'points'})
    [['season', 'round', 'constructor_id', 'constructor_name', 'points']]
    .copy()
)
print(f'2026: {len(standings_2026)} rows, {standings_2026["round"].nunique()} rounds, '
      f'{standings_2026["constructor_id"].nunique()} constructors')
standings_2026.head(10)

2026: 55 rows, 5 rounds, 11 constructors


,season,round,constructor_id,constructor_name,points
0,2026,1,mercedes,Mercedes,43.0
1,2026,1,ferrari,Ferrari,27.0
2,2026,1,mclaren,McLaren,10.0
3,2026,1,red_bull,Red Bull Racing,8.0
4,2026,1,haas,Haas F1 Team,6.0
5,2026,1,rb,Racing Bulls,4.0
6,2026,1,audi,Audi (Sauber),2.0
7,2026,1,alpine,Alpine,1.0
8,2026,1,aston_martin,Aston Martin,0.0
9,2026,1,cadillac,Cadillac,0.0


## 4. Combine and compute convergence metrics

In [5]:
all_standings = pd.concat(
    [standings_2014, standings_2022, standings_2026],
    ignore_index=True
)

convergence_df = convergence_by_round(all_standings, top_n=10)
print(f'Convergence table: {len(convergence_df)} rows')
print(convergence_df[['season','round','era_name','gap_p1_pn','gini']].head(10))

Convergence table: 46 rows
   season  round               era_name  gap_p1_pn    gini
0    2014      1  Hybrid Power Unit Era       33.0  0.6265
1    2014      2  Hybrid Power Unit Era       68.0  0.5887
2    2014      3  Hybrid Power Unit Era      111.0  0.5737
3    2014      4  Hybrid Power Unit Era      154.0  0.5846
4    2014      5  Hybrid Power Unit Era      197.0  0.5987
5    2014      6  Hybrid Power Unit Era      240.0  0.5959
6    2014      7  Hybrid Power Unit Era      258.0  0.5897
7    2014      8  Hybrid Power Unit Era      301.0  0.5842
8    2014      9  Hybrid Power Unit Era      326.0  0.5761
9    2014     10  Hybrid Power Unit Era      366.0  0.5829


## 5. Season summaries

In [6]:
summary = era_comparison_table(convergence_df)
print(summary.to_string(index=False))

 season              era_name  year_in_era  final_round  gap_p1_pn  gap_normalised   gini
   2014 Hybrid Power Unit Era            1           19      701.0           1.000 0.5978
   2022     Ground Effect Era            1           22      716.0           0.989 0.5616
   2026              2026 Era            1            5      185.0           1.000 0.6164


## 6. Export

In [7]:
exp.export(era_convergence_by_round=convergence_df)
print("Export complete")

  Exporting era_convergence_by_round (46 rows)... done.

manifest.json updated (1 file(s) exported).
Export complete
